# 01. Data setup

Convert BioASQ and ArchEHR into one common case format.

In [ ]:
from pathlib import Path
from collections import defaultdict
import html
import json
import os
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests
import pysbd

## 1. Configuration

In [ ]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd] + list(cwd.parents):
        if (path / "data").exists():
            return path
    return cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"
CACHE_DIR = OUTPUT_DIR / "pubmed_cache"

# ArchEHR train set
ARCH_TRAIN_XML = DATA_DIR / "dev" / "archehr-qa.xml"
ARCH_TRAIN_KEY = DATA_DIR / "dev" / "archehr-qa_key.json"

# ArchEHR held-out test set
ARCH_TEST_XML = DATA_DIR / "test" / "archehr-qa.xml"
ARCH_TEST_KEY = DATA_DIR / "test" / "archehr-qa_key.json"

# BioASQ directories
BIOASQ_TRAIN_JSON = DATA_DIR / "bioasq" / "BioASQ-training13b" / "training13b.json"
BIOASQ_TEST_DIR = DATA_DIR / "bioasq" / "Task13BGoldenEnriched"
BIOASQ_TEST_JSONS = [
    BIOASQ_TEST_DIR / f"13B{i}_golden.json"
    for i in range(1, 5)
]

# In BioASQ dataset, "Summary" type questions are used.
BIOASQ_QUESTION_TYPES = {"summary"}

# Minimum proportion of BioASQ snippets that must align successfully.
BIOASQ_MIN_ALIGNMENT = 0.90

# NCBI PubMed EFetch
NCBI_EMAIL = os.getenv("NCBI_EMAIL", "your_email@example.com")
NCBI_TOOL = "msc_dissertation_uq"
NCBI_API_KEY = os.getenv("NCBI_API_KEY")
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

BATCH_SIZE = 100
REQUEST_SLEEP = 0.40
MAX_RETRIES = 5

# Output/cache directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ARTICLE_CACHE = CACHE_DIR / "pubmed_articles.jsonl"

## 2. Shared functions

In [ ]:
# Remove unnecessary spaces and line breaks
def clean_text(text):
    if text is None:
        return ""
    return " ".join(str(text).split())

# Standardise into the form "list[str]" regardless of the input format.
def ensure_string_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value if x is not None and str(x).strip()]
    if isinstance(value, str):
        return [value] if value.strip() else []
    return [str(value)]

# Extract PMID values in PubMed URL
def extract_pmid(value):
    if value is None:
        return None
    m = re.search(r"(\d+)(?:/)?$", str(value).strip())
    return m.group(1) if m else None

# Save cases as JSONL files
def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Load cases from JSONL files
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Check the overlap between snippets and segmented sentences
def positive_overlap(a_start, a_end, b_start, b_end):
    return max(a_start, b_start) < min(a_end, b_end)

def xml_text(element):
    if element is None:
        return ""
    return "".join(element.itertext())

## 3. Load ArchEHR-QA dataset

In [ ]:
# From archehr-qa_key.json, acquire clinician answer and relevance label for each case.
def load_archehr_key(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}

    for item in data:
        answer_rows = item.get("answers", [])

        labels = {
            str(row["sentence_id"]): row.get("relevance")
            for row in answer_rows
        }

        out[str(item["case_id"])] = {
            "clinician_answer": clean_text(
                item.get("clinician_answer", "")
            ),
            "labels": labels,
            "gold_evidence_complete": "answers" in item,
        }

    return out

# Concatenate information related to each patient case
def load_archehr_cases(
    xml_path,
    key_path,
    split,
):

    key_by_case = load_archehr_key(key_path)
    root = ET.parse(xml_path).getroot()

    cases = []

    for case_elem in root.findall("case"):
        source_id = str(case_elem.attrib["id"])
        key_item = key_by_case[source_id]

        patient_question = clean_text(
            xml_text(case_elem.find("patient_narrative"))
        )

        clinician_question = clean_text(
            xml_text(case_elem.find("clinician_question"))
        )

        clinical_specialty = clean_text(
            xml_text(case_elem.find("clinical_specialty"))
        )

        sentences = []

        sentence_root = case_elem.find(
            "note_excerpt_sentences"
        )

        for sent in sentence_root.findall("sentence"):
            sid = str(sent.attrib["id"])

            sentences.append({
                "sentence_id": sid,
                "text": clean_text(xml_text(sent)),
            })

        if key_item["gold_evidence_complete"]:
            gold_ids = [
                str(sent.attrib["id"])
                for sent in sentence_root.findall("sentence")
                if key_item["labels"].get(
                    str(sent.attrib["id"])
                ) == "essential"
            ]
        else:
            gold_ids = []

        cases.append({
            "dataset": "archehr_qa",
            "split": split,
            "case_id": f"archehr_{source_id}",

            "patient_question": patient_question,
            "clinician_question": clinician_question,
            "clinical_specialty": clinical_specialty,
            "question": clinician_question,

            "sentences": sentences,
            "gold_evidence_ids": gold_ids,
            "gold_evidence_complete":
                key_item["gold_evidence_complete"],

            "reference_answers": (
                [key_item["clinician_answer"]]
                if key_item["clinician_answer"]
                else []
            ),
        })

    return cases

archehr_train_cases = load_archehr_cases(
    ARCH_TRAIN_XML,
    ARCH_TRAIN_KEY,
    split="train",
)

archehr_test_cases = load_archehr_cases(
    ARCH_TEST_XML,
    ARCH_TEST_KEY,
    split="test",
)

print("ArchEHR train cases:", len(archehr_train_cases))
print("ArchEHR test cases:", len(archehr_test_cases))

## 4. Load BioASQ dataset

In [ ]:
# Load summary questions from one BioASQ JSON file.
def load_bioasq_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    cases = []
    for q in data["questions"]:
        if q.get("type") not in BIOASQ_QUESTION_TYPES:
            continue

        pmids = []
        for document in q.get("documents", []):
            pmid = extract_pmid(document)
            if pmid and pmid not in pmids:
                pmids.append(pmid)

        cases.append({
            "source_id": str(q["id"]),
            "question": clean_text(q.get("body", "")),
            "pmids": pmids,
            "snippets": list(q.get("snippets", [])),
            "reference_answers": ensure_string_list(q.get("ideal_answer")),
        })

    return cases

# Training questions are used for the primary analysis.
bioasq_train_raw = load_bioasq_json(BIOASQ_TRAIN_JSON)

# Golden questions are kept separate for held-out validation.
bioasq_test_raw = []
for path in BIOASQ_TEST_JSONS:
    bioasq_test_raw.extend(load_bioasq_json(path))

# Collect unique PMIDs required by either split.
all_bioasq_raw = bioasq_train_raw + bioasq_test_raw
all_pmids = sorted({
    pmid
    for case in all_bioasq_raw
    for pmid in case["pmids"]
})

print("BioASQ training summary questions:", len(bioasq_train_raw))
print("BioASQ golden summary questions:", len(bioasq_test_raw))
print("Unique PubMed records:", len(all_pmids))

In [ ]:
# Each PMID is downloaded once and cached.
def parse_pubmed_xml(payload):
    root = ET.fromstring(payload)
    records = {}

    for article in root.findall(".//PubmedArticle"):
        pmid_elem = article.find("./MedlineCitation/PMID")
        if pmid_elem is None or not pmid_elem.text:
            continue

        pmid = pmid_elem.text.strip()
        title = xml_text(article.find("./MedlineCitation/Article/ArticleTitle"))

        sections = []
        for elem in article.findall("./MedlineCitation/Article/Abstract/AbstractText"):
            sections.append({
                "label": elem.attrib.get("Label"),
                "nlm_category": elem.attrib.get("NlmCategory"),
                "text": xml_text(elem),
            })

        records[pmid] = {
            "pmid": pmid,
            "title": title,
            "abstract_sections": sections,
        }

    return records

def fetch_pubmed_once(pmids):
    payload = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }
    if NCBI_API_KEY:
        payload["api_key"] = NCBI_API_KEY

    response = requests.post(
        EFETCH_URL,
        data=payload,
        timeout=(20, 120),
        headers={"Connection": "close"},
    )
    response.raise_for_status()
    return parse_pubmed_xml(response.content)

def fetch_pubmed(pmids):
    last_error = None
    retryable_status = {429, 500, 502, 503, 504}

    for attempt in range(MAX_RETRIES):
        try:
            return fetch_pubmed_once(pmids)

        except requests.exceptions.HTTPError as exc:
            status = exc.response.status_code if exc.response is not None else None

            # Retry temporary NCBI/server errors, but raise other HTTP errors immediately.
            if status not in retryable_status:
                raise

            last_error = exc

            retry_after = None
            if exc.response is not None:
                retry_after = exc.response.headers.get("Retry-After")

            wait = int(retry_after) if retry_after and retry_after.isdigit() else 2 ** attempt
            print(
                f"PubMed HTTP {status}; retry "
                f"{attempt + 1}/{MAX_RETRIES} in {wait}s "
                f"(batch size={len(pmids)})"
            )
            time.sleep(wait)

        except (
            requests.exceptions.ConnectionError,
            requests.exceptions.Timeout,
            requests.exceptions.ChunkedEncodingError,
            ET.ParseError,
        ) as exc:
            last_error = exc
            wait = 2 ** attempt
            print(
                f"PubMed request failed; retry "
                f"{attempt + 1}/{MAX_RETRIES} in {wait}s "
                f"(batch size={len(pmids)})"
            )
            time.sleep(wait)

    # If a batch repeatedly fails, split it into smaller requests.
    if len(pmids) > 10:
        middle = len(pmids) // 2
        left = fetch_pubmed(pmids[:middle])
        right = fetch_pubmed(pmids[middle:])
        return {**left, **right}

    raise RuntimeError(f"PubMed fetch failed for {pmids}") from last_error

article_cache = {x["pmid"]: x for x in load_jsonl(ARTICLE_CACHE)}
missing_pmids = [pmid for pmid in all_pmids if pmid not in article_cache]

for start in range(0, len(missing_pmids), BATCH_SIZE):
    batch = missing_pmids[start:start + BATCH_SIZE]
    article_cache.update(fetch_pubmed(batch))
    save_jsonl(
        [article_cache[k] for k in sorted(article_cache)],
        ARTICLE_CACHE,
    )
    print("Cached:", len(article_cache), "/", len(all_pmids))
    time.sleep(REQUEST_SLEEP)

In [ ]:
# Select the PubMed text that most accurately matches the BioASQ snippet offsets.
def abstract_variants(article):
    sections = [x for x in article["abstract_sections"] if x.get("text")]
    texts = [x["text"] for x in sections]

    labeled = []
    compact = []
    for row in sections:
        label = row.get("label") or row.get("nlm_category")
        if label:
            labeled.append(f"{label}: {row['text']}")
            compact.append(f"{label}:{row['text']}")
        else:
            labeled.append(row["text"])
            compact.append(row["text"])

    variants = [
        " ".join(texts),
        "\n".join(texts),
        "".join(texts),
        " ".join(labeled),
        "\n".join(labeled),
        " ".join(compact),
        "".join(compact),
    ]

    out = []
    for text in variants:
        if text not in out:
            out.append(text)
    return out

def exact_offset_match(text, snippet):
    start = snippet.get("offsetInBeginSection")
    end = snippet.get("offsetInEndSection")
    target = snippet.get("text", "")

    if not isinstance(start, int) or not isinstance(end, int):
        return False
    if 0 <= start <= end <= len(text) and text[start:end] == target:
        return True
    if 0 <= start <= end < len(text) and text[start:end + 1] == target:
        return True
    return False

snippets_by_pmid = defaultdict(list)
for case in all_bioasq_raw:
    for snippet in case["snippets"]:
        pmid = extract_pmid(snippet.get("document"))
        if pmid:
            snippets_by_pmid[pmid].append(snippet)

articles = {}
for pmid, article in article_cache.items():
    variants = abstract_variants(article)
    snippets = [
        s for s in snippets_by_pmid.get(pmid, [])
        if str(s.get("beginSection", "")).lower() == "abstract"
    ]

    if variants:
        abstract = max(
            variants,
            key=lambda text: (
                sum(exact_offset_match(text, s) for s in snippets),
                sum(bool(s.get("text")) and s["text"] in text for s in snippets),
            ),
        )
    else:
        abstract = ""

    articles[pmid] = {
        "title": article.get("title", ""),
        "abstract": abstract,
    }

In [ ]:
# Sentence segmentation
segmenter = pysbd.Segmenter(language="en", clean=False, char_span=True)

def sentence_spans(pmid, section, text):
    if not text or not text.strip():
        return []

    if section == "title":
        start = len(text) - len(text.lstrip())
        end = len(text.rstrip())
        return [{
            "source_id": f"{pmid}:T001",
            "section": "title",
            "text": text[start:end],
            "start": start,
            "end": end,
        }]

    rows = []
    for i, span in enumerate(segmenter.segment(text), 1):
        start = int(span.start)
        end = int(span.end)

        segment = text[start:end]
        left = len(segment) - len(segment.lstrip())
        right = len(segment) - len(segment.rstrip())
        start += left
        end -= right

        if start < end:
            rows.append({
                "source_id": f"{pmid}:A{i:03d}",
                "section": "abstract",
                "text": text[start:end],
                "start": start,
                "end": end,
            })
    return rows

sentence_cache = {}
for pmid, article in articles.items():
    rows = sentence_spans(pmid, "title", article["title"])
    rows += sentence_spans(pmid, "abstract", article["abstract"])
    sentence_cache[pmid] = rows

In [ ]:
# Snippet alignment
def find_span(source, snippet):
    target = snippet.get("text", "")
    start = snippet.get("offsetInBeginSection")
    end = snippet.get("offsetInEndSection")

    if isinstance(start, int) and isinstance(end, int):
        if 0 <= start <= end <= len(source) and source[start:end] == target:
            return start, end
        if 0 <= start <= end < len(source) and source[start:end + 1] == target:
            return start, end + 1

    positions = []
    pos = 0
    while target:
        idx = source.find(target, pos)
        if idx < 0:
            break
        positions.append((idx, idx + len(target)))
        pos = idx + 1

    if positions:
        if isinstance(start, int):
            return min(positions, key=lambda x: abs(x[0] - start))
        return positions[0]

    unescaped = html.unescape(target)
    if unescaped != target:
        idx = source.find(unescaped)
        if idx >= 0:
            return idx, idx + len(unescaped)

    tokens = [x for x in re.split(r"\s+", target.strip()) if x]
    if tokens:
        pattern = r"\s+".join(re.escape(x) for x in tokens)
        matches = [
            (m.start(), m.end())
            for m in re.finditer(pattern, source, flags=re.DOTALL)
        ]
        if matches:
            if isinstance(start, int):
                return min(matches, key=lambda x: abs(x[0] - start))
            return matches[0]

    return None

# Build BioASQ common-schema cases
def build_bioasq_case(raw_case, split):

    # Question, documents and reference answer are required.
    if not raw_case["pmids"] or not raw_case["reference_answers"]:
        return None, "missing_documents_or_reference", 0.0

    # Candidate evidence must contain all referenced PubMed documents.
    if any(pmid not in articles for pmid in raw_case["pmids"]):
        return None, "missing_pubmed_record", 0.0

    source_sentences = []
    for pmid in raw_case["pmids"]:
        source_sentences.extend(sentence_cache.get(pmid, []))

    source_to_case = {}
    sentences = []

    for i, sent in enumerate(source_sentences, 1):
        sid = str(i)
        source_to_case[sent["source_id"]] = sid

        sentences.append({
            "sentence_id": sid,
            "text": sent["text"],
        })

    gold_source_ids = set()
    n_aligned = 0
    n_snippets = len(raw_case["snippets"])

    # Align each gold snippet independently.
    for snippet in raw_case["snippets"]:

        pmid = extract_pmid(snippet.get("document"))
        begin = str(snippet.get("beginSection", "")).lower()
        end = str(snippet.get("endSection", "")).lower()

        if (
            pmid not in articles
            or begin != end
            or begin not in {"title", "abstract"}
        ):
            continue

        resolved = find_span(
            articles[pmid][begin],
            snippet,
        )

        if resolved is None:
            continue

        matched = [
            sent["source_id"]
            for sent in sentence_cache.get(pmid, [])
            if sent["section"] == begin
            and positive_overlap(
                resolved[0],
                resolved[1],
                sent["start"],
                sent["end"],
            )
        ]

        if not matched:
            continue

        n_aligned += 1
        gold_source_ids.update(matched)

    alignment_rate = (
        n_aligned / n_snippets
        if n_snippets
        else 0.0
    )

    # Main analysis requires at least 90% snippet alignment.
    if alignment_rate < BIOASQ_MIN_ALIGNMENT:
        return None, "insufficient_alignment", alignment_rate

    gold_ids = [
        source_to_case[source_id]
        for source_id in gold_source_ids
        if source_id in source_to_case
    ]

    gold_ids = sorted(gold_ids, key=int)

    if not gold_ids:
        return None, "no_gold_sentences", alignment_rate

    case = {
        "dataset": "bioasq_13b",
        "split": split,
        "case_id": f"bioasq13b_{raw_case['source_id']}",

        "patient_question": raw_case["question"],
        "clinician_question": "",
        "clinical_specialty": "",

        "question": raw_case["question"],

        "sentences": sentences,
        "gold_evidence_ids": gold_ids,

        # Complete only when every gold snippet was aligned
        "gold_evidence_complete": alignment_rate == 1.0,

        "reference_answers": raw_case["reference_answers"],
    }

    return case, "eligible", alignment_rate


def build_bioasq_split(raw_cases, split):

    cases = []
    strict_ids = []
    qc_rows = []

    for raw_case in raw_cases:

        case, status, alignment_rate = build_bioasq_case(
            raw_case,
            split=split,
        )

        case_id = f"bioasq13b_{raw_case['source_id']}"

        qc_rows.append({
            "case_id": case_id,
            "status": status,
            "alignment_rate": alignment_rate,
        })

        if case is not None:
            cases.append(case)

            # Complete gold evidence is available only here.
            if alignment_rate == 1.0:
                strict_ids.append(case_id)

    return cases, strict_ids, qc_rows

bioasq_train_cases, train_strict_ids, train_qc_rows = (
    build_bioasq_split(
        bioasq_train_raw,
        split="train",
    )
)

bioasq_test_cases, test_strict_ids, test_qc_rows = (
    build_bioasq_split(
        bioasq_test_raw,
        split="test",
    )
)

print("BioASQ training cases:", len(bioasq_train_cases))
print("BioASQ training strict evidence cases:", len(train_strict_ids))

print("BioASQ test cases:", len(bioasq_test_cases))
print("BioASQ test strict evidence cases:", len(test_strict_ids))

## 5. Save processed data

In [ ]:
# Validate schema
def validate_case(case):
    required = {
        "dataset",
        "split",
        "case_id",
        "patient_question",
        "clinician_question",
        "clinical_specialty",
        "question",
        "sentences",
        "gold_evidence_ids",
        "gold_evidence_complete",
        "reference_answers",
    }

    if set(case) != required:
        raise ValueError(
            f"{case['case_id']}: unexpected schema {set(case)}"
        )

    ids = {
        str(s["sentence_id"])
        for s in case["sentences"]
    }

    if not set(
        map(str, case["gold_evidence_ids"])
    ).issubset(ids):
        raise ValueError(
            f"{case['case_id']}: invalid gold evidence IDs"
        )

    if (
        case["dataset"] == "archehr_qa"
        and not case["clinician_question"]
    ):
        raise ValueError(
            f"{case['case_id']}: missing clinician question"
        )

all_cases = (
    bioasq_train_cases
    + bioasq_test_cases
    + archehr_train_cases
    + archehr_test_cases
)

for case in all_cases:
    validate_case(case)

save_jsonl(
    bioasq_train_cases,
    PROCESSED_DIR / "bioasq_train_cases.jsonl",
)
save_jsonl(
    bioasq_test_cases,
    PROCESSED_DIR / "bioasq_test_cases.jsonl",
)
save_jsonl(
    archehr_train_cases,
    PROCESSED_DIR / "archehr_train_cases.jsonl",
)

save_jsonl(
    archehr_test_cases,
    PROCESSED_DIR / "archehr_test_cases.jsonl",
)

pd.DataFrame(train_qc_rows).to_csv(
    PROCESSED_DIR / "bioasq_train_qc.csv",
    index=False,
)
pd.DataFrame(test_qc_rows).to_csv(
    PROCESSED_DIR / "bioasq_test_qc.csv",
    index=False,
)

pd.DataFrame(
    {"case_id": train_strict_ids}
).to_csv(
    PROCESSED_DIR / "bioasq_train_strict_ids.csv",
    index=False,
)

pd.DataFrame(
    {"case_id": test_strict_ids}
).to_csv(
    PROCESSED_DIR / "bioasq_test_strict_ids.csv",
    index=False,
)

print("Saved:", PROCESSED_DIR)

print(
    "BioASQ training:",
    len(bioasq_train_cases),
)

print(
    "BioASQ test:",
    len(bioasq_test_cases),
)

print(
    "ArchEHR train:",
    len(archehr_train_cases),
)

print(
    "ArchEHR test:",
    len(archehr_test_cases),
)

In [ ]:
print("BioASQ train:", len(bioasq_train_cases))
print("BioASQ test:", len(bioasq_test_cases))
print("ArchEHR train:", len(archehr_train_cases))
print("ArchEHR test:", len(archehr_test_cases))

print(
    "ArchEHR train with gold evidence:",
    sum(case["gold_evidence_complete"] for case in archehr_train_cases),
    "/",
    len(archehr_train_cases),
)
print(
    "ArchEHR test with gold evidence:",
    sum(case["gold_evidence_complete"] for case in archehr_test_cases),
    "/",
    len(archehr_test_cases),
)
print(
    "ArchEHR test with reference answer:",
    sum(bool(case["reference_answers"]) for case in archehr_test_cases),
    "/",
    len(archehr_test_cases),
)